# 第 4 章：ReAct 模式

LangGraph 入门——手写 ReAct graph vs prebuilt `build_react_graph`。

**关键验收点**：两版通过同一道测试题（spec §9.M2）。

## 4.1 ReAct 论文核心

ReAct = **Reasoning** + **Acting** 交替循环：

1. **思考**（Reasoning）：LLM 分析当前状态
2. **行动**（Acting）：调工具获取新信息
3. **观察**（Observation）：工具返回的结果
4. 循环 1-3 直到得出最终答案

类比前端：
- StateGraph ≈ Redux store + reducer
- add_node ≈ 注册 reducer function
- add_conditional_edges ≈ switch 语句
- add_messages reducer ≈ Redux reducer（合并 state）


In [ ]:
# 手写 ReAct graph
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

from agent_core.state import ReActState


def _should_continue(state: ReActState) -> str:
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END


builder = StateGraph(ReActState)
builder.add_node("agent", ...)  # LLM node
builder.add_node("tools", ToolNode(...))  # 工具 node
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", _should_continue)
builder.add_edge("tools", "agent")
graph = builder.compile()

## 4.2 手写 vs prebuilt 对比

| 方面 | 手写版 | prebuilt 版 |
|---|---|---|
| 调用方式 | 自建 StateGraph | `build_react_graph(llm, tools)` |
| 理解深度 | 看清每一步 | 封装了标准模式 |
| 等价性 | **完全等价** | **完全等价** |

就像手写 Redux reducer vs `createReducer()`——两者产出相同 state。


In [ ]:
from agent_core import build_react_graph
from hand_coded_react import build_hand_coded_react
from prebuilt_react import build_prebuilt_react


# 三种方式产出等价的 graph
# graph_hand = build_hand_coded_react(llm=llm, tools=[add])
# graph_pre = build_prebuilt_react(llm=llm, tools=[add])
# graph_core = build_react_graph(llm=llm, tools=[add])


# 验收: 对同一 input, 三版产出相同 final answer
# (详见 tests/test_cross_implementation.py)